In [7]:
import json
from tqdm import tqdm
import pandas as pd

pd.set_option("max_colwidth", 400)

In [2]:
predictions_file = "../data/results/2025-11-25_19-49-41/predictions.jsonl"
ground_truth_file = "../data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"

In [3]:
# when merged with PR , can use the following code to load

#import os
#from typing import Hashable
#from kibad_llm.dataset.json import read_and_preprocess
#from kibad_llm.dataset.utils import merge_references_into_predictions

#def load(file: str) -> dict[Hashable, dict[str, dict]]:
#    predictions = read_and_preprocess(
#        id_key="file_name",
#        file=file,
#        process_id=lambda x: os.path.splitext(x)[0],
#        preprocess=lambda x: x.get("structured", None),
#    )
#    references = read_and_preprocess(
#        id_key="zotitem_ptr_id",
#        file="data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl",
#    )
#    result = merge_references_into_predictions(predictions, references)
#    return result
#dataset = load(predictions_file)

In [4]:
# load JSONL files for ground truth and predictions
with open(predictions_file, "rb") as p_in:
    predictions = [json.loads(line) for line in p_in]

with open(ground_truth_file, "rb") as g_in:
    ground_truth = [json.loads(line) for line in g_in]

In [5]:
# collect and sort the zotero file ids
predictions_zotero_keys = sorted([p["file_name"][:8] for p in predictions])
predictions = sorted(predictions, key=lambda x: x["file_name"])
# filter ground truth file to retain only entries for which we have predictions
ground_truth_filtered = sorted(
    [g for g in ground_truth if g["zotitem_ptr_id"] in predictions_zotero_keys],
    key=lambda x: x["zotitem_ptr_id"],
)

In [10]:
# change structure of predictions to 'flatten' it to a list of keys where keys
# are the schema types we extract, plus doc id
predictions_flat = [dict(p["structured"] or {}) for p in predictions]
for idx, p in enumerate(predictions_flat):
    p.update({"id": predictions[idx]["file_name"][:8]})

# change structure of ground truth to retain only predicted schema types plus id
ground_truth_filtered_flat = [
    {k: v for k, v in g.items() if k in predictions_flat[0].keys()} for g in ground_truth_filtered
]
for idx, g in enumerate(ground_truth_filtered_flat):
    g.update({"id": ground_truth_filtered[idx]["zotitem_ptr_id"]})


def flatten(prediction, key_name, flattened_keys):

    if key_name in prediction and prediction[key_name] and len(prediction[key_name]) > 0:
        flattened_lists = [
            [e[k[len(key_name) + 1:]] for e in prediction[key_name]] for k in flattened_keys
        ]
    else:
        flattened_lists = [[] for _ in flattened_keys]
    return flattened_lists

# use flatten from +metric.flatten_dicts=true param
# change structure of nested / compound types to have flat lists per subtype
compounds = {
    "taxa": ["taxa.german_name", "taxa.scientific_name", "taxa.species_group"],
    "direct_driver": ["direct_driver.category", "direct_driver.details"],
    "indirect_driver": ["indirect_driver.category", "indirect_driver.details"],
    "soil": ["soil.name", "soil.depth"],
    "ecosystem_service": [
        "ecosystem_service.category",
        "ecosystem_service.term",
        "ecosystem_service.details",
    ],
    "management_measure": ["management_measure.description", "management_measure.success"],
    "conservation_area": [
        "conservation_area.name",
        "conservation_area.description",
        "conservation_area.success",
    ],
    "impulse_measure": ["impulse_measure.description", "impulse_measure.success"],
    "location": ["location.country", "location.federal_state", "location.name"],
    "ecosystem_type": [
        "ecosystem_type.category",
        "ecosystem_type.term",
        "ecosystem_type.description",
    ],
}
for p in tqdm(predictions_flat):
    for c in compounds.keys():
        flattened_lists = flatten(p, c, compounds[c])
        p.update(dict(zip(compounds[c], flattened_lists)))
        if c in p:
            del p[c]

for g in tqdm(ground_truth_filtered_flat):
    for c in compounds.keys():
        flattened_lists = flatten(g, c, compounds[c])
        g.update(dict(zip(compounds[c], flattened_lists)))
        if c in g:
            del g[c]

df_predictions = pd.DataFrame.from_dict(predictions_flat)
df_ground_truth = pd.DataFrame.from_dict(ground_truth_filtered_flat)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 39401.63it/s]


In [ ]:
from typing import Hashable, Tuple

def _as_list_or_empty(x):
    """
    Wandelt x in eine Liste um:
    - Wenn x eine Liste/Tuple/Set ist → Liste zurück
    - Wenn x NaN/None → leere Liste
    - Wenn x ein String oder Skalar → [x]
    """
    import numpy as np
    import pandas as pd

    if x is None:
        return []
    # Series oder np.ndarray
    if isinstance(x, (pd.Series, np.ndarray)):
        x_ret = list(x)
        return [e for e in x_ret if e is not None]
    # Liste/Tuple/Set → Liste
    if isinstance(x, (list, tuple, set)):
        x_ret = list(x)
        return [e for e in x_ret if e is not None and e != '']
    # alles andere (z.B. String, Zahl) → einzelnelement-Liste
    if x == '':
        return []
    return [x]



def align_row(pred_list, true_list, missing_label = '_NA_'):
    """
    Für eine Zeile: pred_list und true_list sind (möglicherweise) Listen von Strings.
    Gibt (union_sorted, aligned_pred, aligned_true) zurück:
      - union_sorted: alphabetisch sortierte Union der Labels
      - aligned_pred: Liste gleicher Länge; Label wenn in pred_list vorhanden, sonst None
      - aligned_true: analog für true_list
    """
    pred = [str(v) for v in _as_list_or_empty(pred_list)]
    true = [str(v) for v in _as_list_or_empty(true_list)]

    union = sorted(set(pred) | set(true))
    aligned_pred = [label if label in pred else missing_label for label in union]
    aligned_true = [label if label in true else missing_label for label in union]

    return union, aligned_pred, aligned_true

# -------------------------
# Beispiel-Anwendung: df_predictions und df_ground_truth haben dieselbe Reihenfolge / Index
# -------------------------

# Beispiel-DataFrames (nur zur Demonstration)
# df_predictions = pd.DataFrame({'taxa.german_name': [["Amsel","Meise"], ["Kohlmeise"], []]})
# df_ground_truth = pd.DataFrame({'taxa.german_name': [["Amsel"], ["Meise","Kohlmeise"], ["Buntspecht"]]})

# Sicherstellen, dass die Indizes zusammenpassen; ansonsten vorher joinen/merge
if not df_predictions.index.equals(df_ground_truth.index):
    # wenn nicht, joinieren wir per default inner-join auf Index (alternative: reset_index+merge, oder auf ID-Spalte joinen)
    df = df_predictions[['taxa.german_name']].rename(columns={'taxa.german_name':'y_pred'}) \
         .join(df_ground_truth[['taxa.german_name']].rename(columns={'taxa.german_name':'y_true'}),
               how='inner')
else:
    df = pd.DataFrame({
        'y_pred': df_predictions['taxa.german_name'],
        'y_true': df_ground_truth['taxa.german_name']
    })

# Zeilenweise anwenden und neue Spalten anlegen
res = df.apply(lambda row: align_row(row['y_pred'], row['y_true']), axis=1)

# res ist eine Series mit Tupeln (union, aligned_pred, aligned_true) — wir entpacken:
df[['union_labels', 'aligned_pred', 'aligned_true']] = pd.DataFrame(res.tolist(), index=res.index)

# Ergebnis: df enthält nun pro Zeile die union und die beiden aligned-Listen
# Beispiel-Ausgabe:
# df[['union_labels','aligned_pred','aligned_true']].head()

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from itertools import chain

# Flatten der Listen-Spalten
union_labels_flat = sorted(list(set(list(chain.from_iterable(df['union_labels'])))))
#print(union_labels_flat)
y_pred_flat = list(chain.from_iterable(df['aligned_pred']))
y_true_flat = list(chain.from_iterable(df['aligned_true']))

# F1-Score berechnen (für Multiclass)
score = precision_recall_fscore_support(y_true_flat, y_pred_flat, average='micro', labels=union_labels_flat)  # oder 'micro', 'weighted'
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=union_labels_flat + ['_NA_'])
print("F1-Score:", score)
import matplotlib.pyplot as plt
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=union_labels_flat + ['_NA_'])
disp.plot()

plt.show()

F1-Score: (0.23943661971830985, 0.1471861471861472, 0.18230563002680966, None)


In [ ]:
pd.concat(
    [
        df_ground_truth["id"],
        df_ground_truth["taxa.german_name"],
        df_predictions["taxa.german_name"],
    ],
    axis=1,
)

In [ ]:
# biodiversity_variable -> gold label sind oft Englisch, z.B. 'Abundance of mice', 'Abundance of voles'
# transformation_potential -> nie predicted
# in location sind gold werte wie '(('country', 'Australia'), ('federal_state', 'Baden-Württemberg'), ('name', 'Western Australian coast'))' ???
# in location sind gold werte wie (('country', 'Belgium'),) |   (('country', 'Cambodia'), ('name', 'Cambodia'))
# species mismatches wegen kleiner unterschiede, z.B predicted = ('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium L.'), ('species_group', 'Gefäßpflanzen')
# gold = (('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium'), ('species_group', 'Gefäßpflanzen')) -> einziger Unterschied ist ' L.' im scientific name
# ansonsten sehen predictions für taxa eigentlich gut aus (immer deutscher oder lateinischer Begriff)

In [ ]:
df_predictions